In [4]:
from joblib import load
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Text, Button, VBox, HBox, Output
from IPython.display import display, clear_output

In [5]:
# Step 1: Load data
aggregated_metrics = load("/Users/hanwu/ML/AnalogDesignAuto_MultiAgent/custom_env/analysis_file/sum.joblib")  # Adjust the path as necessary

# Keep a copy of the original data with indexes
original_data_with_index = [{'index': i, **result} for i, result in enumerate(aggregated_metrics)]

# Extract result dict in each element and keep original indexes
aggregated_metrics = [{'index': item['index'], **item['result']} for item in original_data_with_index]

# Data cleaning
df = pd.DataFrame(aggregated_metrics).set_index('index')  # Use 'index' as the DataFrame index
print(f"Original data groups: {df.shape[0]}")

# Dynamic data cleaning logic
for column in df.columns:
    original_count = df.shape[0]
    # Skip the 'original_index' column
    if column == 'index':
        continue
    if column == 'pwr':
        # Delete rows where 'pwr' is 1
        df = df[df[column] != 1]
    else:
        # Delete rows where other columns are 0
        df = df[df[column] != 0]
    cleaned_count = df.shape[0]
    print(f"Deleted {original_count - cleaned_count} rows from '{column}' due to cleaning criteria.")

# Print the original and cleaned data counts for comparison
print(f"Cleaned data groups: {df.shape[0]}")

Original data groups: 3628
Deleted 697 rows from 'phaseMargin' due to cleaning criteria.
Deleted 0 rows from 'gainBandWidth' due to cleaning criteria.
Deleted 3 rows from 'pwr' due to cleaning criteria.
Cleaned data groups: 2928


In [6]:
widgets_dict = {}
for column in df.columns:
    widgets_dict[column] = {
        'min_text': Text(value=str(df[column].min()), description=f'{column} Min:'),
        'max_text': Text(value=str(df[column].max()), description=f'{column} Max:'),
        'update_button': Button(description=f'Update {column}'),
        'reset_button': Button(description='Reset'),
        'output': Output()
    }
print_button = Button(description='Print Selected Data')

def plot_distribution(column, data):
    with widgets_dict[column]['output']:
        clear_output(wait=True) 
        plt.hist(data, bins=30, alpha=0.75)
        plt.title(f'{column} Distribution')
        plt.xlabel(column)
        plt.ylabel('Frequency')
        plt.show()
        
def reset_view(b):
    for col in df.columns:
        widgets_dict[col]['min_text'].value = str(df[col].min())
        widgets_dict[col]['max_text'].value = str(df[col].max())
        plot_distribution(col, df[col])  

def update_all(b):
    query_str = ' & '.join([f'({col} >= {widgets_dict[col]["min_text"].value}) & ({col} <= {widgets_dict[col]["max_text"].value})' for col in df.columns])
    filtered_df = df.query(query_str)
    
    for col in filtered_df.columns:
        plot_distribution(col, filtered_df[col])
        
def print_selected_data(b):
    query_str = ' & '.join([f'({col} >= {widgets_dict[col]["min_text"].value}) & ({col} <= {widgets_dict[col]["max_text"].value})' for col in df.columns])
    filtered_df = df.query(query_str)
    
    selected_indexes = filtered_df.index.tolist()
    selected_original_data = [original_data_with_index[i] for i in selected_indexes]
    
    for item in selected_original_data:
        print(f"Index: {item['index']}, Params: {item.get('param', {})}, Results: {item['result']}")

for column, widget in widgets_dict.items():
    widget['update_button'].on_click(update_all)
    widget['reset_button'].on_click(reset_view)
    display(VBox([HBox([widget['min_text'], widget['max_text'], widget['update_button'], widget['reset_button']]), widget['output']]))
    plot_distribution(column, df[column])
    
print_button.on_click(print_selected_data)
    
display(print_button)

Button(description='Print Filtered Data', style=ButtonStyle())